In [ ]:
import numpy as np
import pandas as pd
from scipy.io import loadmat

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import seaborn as sns
from pathlib import Path

Path("Figures").mkdir(exist_ok=True)


# Load data

In [ ]:
dataPath = '../benchmarkAnalysis/mergeResultsTables'
merged_file = dataPath + '/allMergedResults_V2_2_py.mat'

mat_data = loadmat(merged_file)
S = mat_data["S"]
n_rows = S.shape[0]

data = []
for i in range(n_rows):
    row_struct = S[i, 0]
    row = {}
    for field in S.dtype.names:
        value = row_struct[field]
        if isinstance(value, np.ndarray):
            if value.shape == (1, 1):  # scalar
                value = value[0, 0]
            elif value.dtype.kind in {"U", "S"} and value.shape[0] == 1:
                value = ''.join(value[0])
            elif value.dtype.char == "S":  # bytes from MATLAB char
                try:
                    value = b''.join(value.flat).decode("utf-8")
                except Exception:
                    value = str(value)
        row[field] = value
    data.append(row)

df = pd.DataFrame(data)

print(df.columns)
print(df[["projectName", "templateName"]].head())



# Plot settings

In [ ]:
fontsize = 10
ax_labelsize = 9

plt.rcParams.update({
    'font.size': fontsize,
    'text.usetex': True,
    'font.family': 'serif',
    'axes.labelsize': fontsize,
    'axes.titlesize': fontsize,
    'legend.fontsize': fontsize,
    'xtick.labelsize': ax_labelsize,
    'ytick.labelsize': ax_labelsize,
    'lines.markersize': 4
})

single_width_mm = 86 # in mm
double_width_mm = 178 # in mm
fig_width_mm = double_width_mm

mm2in = 1 / 25.4


In [ ]:

flierprops = {"marker": "x", "markersize": 4}
bp_linewidth = 0.75*1.5
bp_linecolor = "black"


# Supplementary Figures S1 and S2

In [ ]:
def compute_percentile_gap_df(
    df,
    percentile=50,
    chi2_col="chi2_sorted",
    project_col="projectName",
    n_data_col="nDataPoints",
    n_params_col="nParams",
):
    """
    For each benchmark, compute:

        gap = (chi2_percentile - chi2_best) / dof

    where:

        dof = n_data - n_params

    """

    rank = int(percentile)
    rows = []

    for _, row in df.iterrows():
        chi2 = np.asarray(row[chi2_col]).ravel()

        if chi2.size == 0:
            gap = np.nan
        else:
            dof = row[n_data_col] - row[n_params_col]

            if dof <= 0:
                gap = np.nan
            else:
                chi2_best = chi2[0]
                chi2_percentile = chi2[rank - 1]

                if not np.isfinite(chi2_percentile):
                    gap = np.nan
                else:
                    gap = np.log10((chi2_percentile - chi2_best) / dof)

        rows.append(
            {
                project_col: row[project_col],
                f"gap_p{percentile}_per_dof": gap,
            }
        )

    return pd.DataFrame(rows)


In [ ]:
def compute_recurrent_objective_clusters_df(
    df,
    m=2,
    eps=1e-4,
    tau=np.inf,
    chi2_col="chi2_sorted",
    project_col="projectName",
    n_data_col="nDataPoints",
    n_params_col="nParams",
):
    rows = []

    for _, row in df.iterrows():
        chi2 = np.asarray(row[chi2_col]).ravel()
        dof = row[n_data_col] - row[n_params_col]

        if chi2.size == 0 or dof <= 0:
            n_clusters = np.nan
            cluster_sizes = []
            entropy = np.nan
        else:
            chi2_best = chi2[0]

            gaps_to_best = (chi2 - chi2_best) / dof
            chi2 = chi2[gaps_to_best <= tau]

            if chi2.size == 0:
                n_clusters = 0
                cluster_sizes = []
                entropy = 0.0
            else:
                clusters = []
                current_cluster = [chi2[0]]

                for value in chi2[1:]:
                    step = (value - current_cluster[-1]) / dof

                    if step <= eps:
                        current_cluster.append(value)
                    else:
                        clusters.append(current_cluster)
                        current_cluster = [value]

                clusters.append(current_cluster)

                recurrent_clusters = [
                    cluster for cluster in clusters
                    if len(cluster) >= m
                ]

                cluster_sizes = np.array(
                    [len(cluster) for cluster in recurrent_clusters],
                    dtype=float,
                )

                n_clusters = len(recurrent_clusters)

                if cluster_sizes.size == 0:
                    entropy = 0.0
                    cluster_sizes = []
                else:
                    p = cluster_sizes / cluster_sizes.sum()
                    entropy = -np.sum(p * np.log(p))
                    cluster_sizes = cluster_sizes.astype(int).tolist()

        rows.append(
            {
                project_col: row[project_col],
                f"n_recurrent_clusters_m{m}": n_clusters,
                f"cluster_entropy_m{m}": entropy,
                f"cluster_sizes_m{m}": cluster_sizes,
            }
        )

    return pd.DataFrame(rows)


In [ ]:
def safe_merge(left, right, on):
    overlap = set(left.columns) & set(right.columns) - set([on])
    if overlap:
        left = left.drop(columns=list(overlap))
    return left.merge(right, on=on)

metrics_df = compute_percentile_gap_df(df, percentile=50)

tmp_df = compute_percentile_gap_df(df, percentile=10)
metrics_df = safe_merge(metrics_df, tmp_df, on="projectName")

tmp_df = compute_percentile_gap_df(df, percentile=90)
metrics_df = safe_merge(metrics_df, tmp_df, on="projectName")

tmp_df = compute_recurrent_objective_clusters_df(df, m=1, eps=1e-2)
metrics_df = safe_merge(metrics_df, tmp_df, on="projectName")

tmp_df = compute_recurrent_objective_clusters_df(df, m=2, eps=1e-2)
metrics_df = safe_merge(metrics_df, tmp_df, on="projectName")

tmp_df = compute_recurrent_objective_clusters_df(df, m=4, eps=1e-2)
metrics_df = safe_merge(metrics_df, tmp_df, on="projectName")

df = df.drop(columns=[c for c in metrics_df.columns if c in df.columns and c != "projectName"])
df = df.merge(metrics_df, on="projectName")

metrics_df


In [ ]:
templates = df[df["qTemplate"] == 1]
synthetics = df[df["qTemplate"] == 0]

def fit_value_counts(dataframe, chi2_col="chi2_sorted"):
    max_len = int(dataframe[chi2_col].apply(lambda x: len(np.asarray(x).ravel())).max())
    fit_idx = np.arange(1, max_len + 1)

    counts = []
    for i in fit_idx:
        valid = dataframe[chi2_col].apply(
            lambda x: len(np.asarray(x).ravel()) >= i and np.isfinite(np.asarray(x).ravel()[i - 1])
        ).sum()
        counts.append(valid)

    return fit_idx, np.asarray(counts)

x_t, y_t = fit_value_counts(templates)
x_s, y_s = fit_value_counts(synthetics)



In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(double_width_mm * mm2in, 130 * mm2in), constrained_layout=True)

template_color = sns.color_palette("deep")[0]
synthetic_color = sns.color_palette("deep")[1]

top_vars = ['gap_p10_per_dof', 'gap_p50_per_dof', 'gap_p90_per_dof']
top_labels = ['B) 10th Best Fit', 'C) 50th Best Fit', 'D) 90th Best Fit']

for j, (var, label) in enumerate(zip(top_vars, top_labels)):
    ax = axes[1, j]

    records = []
    for _, row in df.iterrows():
        if pd.isna(row[var]):
            continue
        if row["qTemplate"] == 1:
            records.append({"group": "all templates", "variable": var, "value": row[var]})
        else:
            records.append({"group": "all synthetic", "variable": var, "value": row[var]})
    plot_df = pd.DataFrame(records)

    [nSynth, nTemp] = plot_df['group'].value_counts()

    plot_df = pd.DataFrame(records)

    sns.boxplot(
        x="variable", y="value", hue="group",
        data=plot_df, ax=ax,
        legend=False, log_scale=False,
        linewidth=bp_linewidth, linecolor=bp_linecolor,
        flierprops={"marker": "x", "markersize": 0}
    )
    sns.stripplot(
        x="variable", y="value", hue="group",
        data=plot_df, ax=ax,
        legend=False, log_scale=False,
        size=3, dodge=True,
        palette=['0.25', 'none']
    )
    sns.stripplot(
        x="variable", y="value", hue="group",
        data=plot_df, ax=ax,
        legend=False, log_scale=False,
        size=1.5, dodge=True,
        palette=['none', '0.25']
    )

    ax.set_title(label)
    ax.set_xlabel("")
    ax.set_ylabel("" if j > 0 else "log10($\\Delta\\chi^2$/dof)")
    ax.set_xticks([])
    ax.grid(True, linestyle="--", alpha=0.5)

    handles = [
        mpatches.Patch(color=sns.color_palette("deep")[0], label=f"Templates ({nTemp}/22)", ls='-'),
        mpatches.Patch(color=sns.color_palette("deep")[1], label=f"Synthetic ({nSynth}/1100)", linewidth=1)
    ]
    by_label = {h.get_label(): h for h in handles}
    leg = ax.legend(by_label.values(), by_label.keys(),
                    loc="lower center", ncol=1,
                    bbox_to_anchor=(0.55, -0.5),
                    )   




gs = axes[0, 0].get_gridspec()
for a in axes[0, :]:
    fig.delaxes(a)
ax1 = fig.add_subplot(gs[0, :])
ax2 = ax1.twinx()

l1 = ax1.plot(x_t, y_t, marker="o", lw=1.5, color=template_color, label="Templates")
l2 = ax2.plot(x_s, y_s, marker="^", lw=1.5, color=synthetic_color, label="Synthetic")

ax1.set_xlim(1, 100)
ax1.set_xticks(np.arange(0, 110, 10))
ax1.set_xlabel("Fit index i")
ax1.set_ylabel("Templates with numeric fit value", color=template_color)
ax2.set_ylabel("Synthetic problems with numeric fit value", color=synthetic_color)
ax1.grid(True, linestyle="--", alpha=0.5)

for x_pos, label_idx in zip([10, 50, 90], range(3)):
    ax1.axvline(x_pos, color="black", linestyle="--", linewidth=0.8, alpha=0.7)

    lines = l1 + l2
    labels = [line.get_label() for line in lines]
    ax1.legend(lines, labels, loc="lower left")

ax1.set_title("A) Availability of i-th Best Fit")
ax1.tick_params(axis='y')
ax2.tick_params(axis='y')
ax1.set_ylabel("Number of Templates", c='k')
ax2.set_ylabel("Number of Synthetics", c='k')

ticks = np.array([6, 10, 14, 18, 22])
ax1.set_yticks(ticks)
ax2.set_yticks(ticks * (1100 / 22))

ax1.set_ylim(min(ticks), max(y_t) * 1.05)
ax2.set_ylim(min(ticks * (1100 / 22)), max(y_s) * 1.05)

axes[1, 0].set_ylim(-16, 16)
axes[1, 0].set_yticks([-16, -8, 0, 8, 16])
axes[1, 1].set_ylim(-4, 4)
axes[1, 1].set_yticks([-4, -2, 0, 2, 4])
axes[1, 2].set_ylim(-2, 2)
axes[1, 2].set_yticks([-2, -1, 0, 1, 2])


plt.tight_layout()
plt.savefig("Figures/S1_StripBoxPlots_Multistart_Fit-Differences_Extended.pdf", bbox_inches='tight', bbox_extra_artists=(leg, ))
plt.show()


In [ ]:
from matplotlib.ticker import FormatStrFormatter

num_vars = ['n_recurrent_clusters_m1', 'n_recurrent_clusters_m2', 'n_recurrent_clusters_m4']
ent_vars = ['cluster_entropy_m1', 'cluster_entropy_m2', 'cluster_entropy_m4']
col_labels = ['All Levels ($\\geq1$ times)', 'Repeated Levels ($\\geq2$ times)', 'Frequent Levels ($\\geq4$ times)']

fig, axes = plt.subplots(2, 3, figsize=(double_width_mm*mm2in, 80*mm2in),
                         constrained_layout=True)
axes = axes.reshape(2, 3)

for j, (nvar, envar, label) in enumerate(zip(num_vars, ent_vars, col_labels)):
    ax = axes[0, j]
    records = []
    for _, row in df.iterrows():
        val = row.get(nvar, np.nan)
        if pd.isna(val):
            continue
        grp = "all templates" if row["qTemplate"] == 1 else "all synthetic"
        records.append({"group": grp, "variable": nvar, "value": val})
    plot_df = pd.DataFrame(records)
    nSynth = int(plot_df['group'].value_counts().get('all synthetic', 0))
    nTemp = int(plot_df['group'].value_counts().get('all templates', 0))

    flierprops_temp = {"marker": "x", "markersize": 0}
    sns.boxplot(x="variable", y="value", hue="group",
                data=plot_df, ax=ax,
                legend=False,
                linewidth=bp_linewidth, color=None,
                flierprops=flierprops_temp)
    sns.stripplot(x="variable", y="value", hue="group",
                  data=plot_df, ax=ax,
                  legend=False, size=3, dodge=True,
                  palette=['0.25', 'none'])
    sns.stripplot(x="variable", y="value", hue="group",
                  data=plot_df, ax=ax,
                  legend=False, size=1.5, dodge=True,
                  palette=['none', '0.25'])

    ax.set_title(label)
    ax.set_xticks([])
    ax.set_xlabel("")
    ax.set_ylabel("Level Count" if j == 0 else "")
    ax.grid(True, linestyle="--", alpha=0.5)

    ax = axes[1, j]
    records = []
    for _, row in df.iterrows():
        val = row.get(envar, np.nan)
        if pd.isna(val):
            continue
        grp = "all templates" if row["qTemplate"] == 1 else "all synthetic"
        records.append({"group": grp, "variable": envar, "value": val})
    plot_df = pd.DataFrame(records)

    if plot_df.empty:
        nSynth = nSynth if 'nSynth' in locals() else 0
        nTemp = nTemp if 'nTemp' in locals() else 0
    else:
        nSynth = int(plot_df['group'].value_counts().get('all synthetic', 0))
        nTemp = int(plot_df['group'].value_counts().get('all templates', 0))

    sns.boxplot(x="variable", y="value", hue="group",
                data=plot_df, ax=ax,
                legend=False,
                linewidth=bp_linewidth, flierprops=flierprops_temp)
    sns.stripplot(x="variable", y="value", hue="group",
                  data=plot_df, ax=ax,
                  legend=False, size=3, dodge=True,
                  palette=['0.25', 'none'])
    sns.stripplot(x="variable", y="value", hue="group",
                  data=plot_df, ax=ax,
                  legend=False, size=1.5, dodge=True,
                  palette=['none', '0.25'])

    ax.set_xticks([])
    ax.set_xlabel("")
    ax.set_ylabel("Occupancy Entropy" if j == 0 else "")
    ax.grid(True, linestyle="--", alpha=0.5)


def range_with_margin(values, margin=0.1):
    if len(values) == 0:
        return (0, 1)
    vmin, vmax = min(values), max(values)
    if vmin == vmax:
        return (vmin - margin, vmax + margin)
    else:
        return (vmin - margin * (vmax - vmin), vmax + margin * (vmax - vmin))

axes[0,0].set_ylim(range_with_margin([0, 100]))
axes[0,1].set_ylim(range_with_margin([0, 25]))
axes[0,2].set_ylim(range_with_margin([0, 12.5]))

axes[0,0].set_yticks([0, 50, 100])

axes[1,0].set_ylim(range_with_margin([0, 4.6]))
axes[1,1].set_ylim(range_with_margin([0, 4.6]))
axes[1,2].set_ylim(range_with_margin([0, 4.6]))

for ax in axes[0,:]:
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.0f'))

for ax in axes[1,:]:
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

handles = [
    mpatches.Patch(color=sns.color_palette("deep")[0], label=f"Templates ({nTemp}/22)"),
    mpatches.Patch(color=sns.color_palette("deep")[1], label=f"Synthetic ({nSynth}/1100)")
]
by_label = {h.get_label(): h for h in handles}
fig.legend(by_label.values(), by_label.keys(), loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.15))

for idx, ax in enumerate(axes.flat):
    ax.text(0.04, 0.96, chr(65 + idx) + ')',
            transform=ax.transAxes, 
            va='top', ha='left')


plt.savefig("Figures/S2_StripBoxPlots_Multistart_Recurrency.pdf", bbox_inches='tight')

plt.show()


# Supplementary Figures S3A and S3B

In [ ]:
import statsmodels.api as sm
from matplotlib.ticker import FormatStrFormatter

definitions = pd.read_csv("data/observable_definitions.csv")
existing_outcomes = pd.read_csv("data/existing_outcomes.csv")
outcome_columns = [
    "fraction_identifiable_dynamic",
    "fraction_global_optimum",
]


In [ ]:
fraction_predictors = (
    definitions
    .assign(
        has_sum=definitions["core_expression"].str.contains("+", regex=False)
    )
    .groupby(["template_name", "benchmark_name"], as_index=False)
    .agg(
        n_observables=("observable_name", "size"),
        n_sums=("has_sum", "sum"),
        n_with_scale=("has_scale", "sum"),
        n_with_offset=("has_offset", "sum"),
    )
)

fraction_predictors["fraction_sums"] = (
    fraction_predictors["n_sums"]
    / fraction_predictors["n_observables"]
)
fraction_predictors["fraction_scales_offsets"] = (
    (
        fraction_predictors["n_with_scale"]
        + fraction_predictors["n_with_offset"]
    )
    / fraction_predictors["n_observables"]
)

fraction_analysis_data = fraction_predictors.merge(
    existing_outcomes,
    on="benchmark_name",
    how="inner",
    validate="one_to_one",
)



In [ ]:
fraction_predictor_columns = [
    "fraction_sums",
    "fraction_scales_offsets",
    "n_observables",
]

fraction_regression_result_columns = [
    "template_name",
    "outcome",
    "n_problems",
    "fraction_sums_coefficient",
    "fraction_sums_ci_lower",
    "fraction_sums_ci_upper",
    "fraction_sums_p_value",
    "fraction_scales_offsets_coefficient",
    "fraction_scales_offsets_ci_lower",
    "fraction_scales_offsets_ci_upper",
    "fraction_scales_offsets_p_value",
    "n_observables_coefficient",
    "n_observables_ci_lower",
    "n_observables_ci_upper",
    "n_observables_p_value",
    "r_squared",
    "status",
]

def fit_fraction_regression(template_data, outcome):
    fit_data = template_data[
        [outcome] + fraction_predictor_columns
    ].dropna()

    result = {
        "template_name": template_data["template_name"].iloc[0],
        "outcome": outcome,
        "n_problems": len(fit_data),
        "fraction_sums_coefficient": np.nan,
        "fraction_sums_ci_lower": np.nan,
        "fraction_sums_ci_upper": np.nan,
        "fraction_sums_p_value": np.nan,
        "fraction_scales_offsets_coefficient": np.nan,
        "fraction_scales_offsets_ci_lower": np.nan,
        "fraction_scales_offsets_ci_upper": np.nan,
        "fraction_scales_offsets_p_value": np.nan,
        "n_observables_coefficient": np.nan,
        "n_observables_ci_lower": np.nan,
        "n_observables_ci_upper": np.nan,
        "n_observables_p_value": np.nan,
        "r_squared": np.nan,
        "status": "insufficient_data",
    }

    if len(fit_data) < 4:
        return result
    if fit_data[[outcome] + fraction_predictor_columns].nunique().le(1).any():
        result["status"] = "no_variation"
        return result

    design_matrix = sm.add_constant(
        fit_data[fraction_predictor_columns],
        has_constant="add",
    )
    fitted_model = sm.OLS(fit_data[outcome], design_matrix).fit()
    confidence_intervals = fitted_model.conf_int()

    result.update({
        "fraction_sums_coefficient": fitted_model.params["fraction_sums"],
        "fraction_sums_ci_lower": confidence_intervals.loc["fraction_sums", 0],
        "fraction_sums_ci_upper": confidence_intervals.loc["fraction_sums", 1],
        "fraction_sums_p_value": fitted_model.pvalues["fraction_sums"],
        "fraction_scales_offsets_coefficient": fitted_model.params["fraction_scales_offsets"],
        "fraction_scales_offsets_ci_lower": confidence_intervals.loc["fraction_scales_offsets", 0],
        "fraction_scales_offsets_ci_upper": confidence_intervals.loc["fraction_scales_offsets", 1],
        "fraction_scales_offsets_p_value": fitted_model.pvalues["fraction_scales_offsets"],
        "n_observables_coefficient": fitted_model.params["n_observables"],
        "n_observables_ci_lower": confidence_intervals.loc["n_observables", 0],
        "n_observables_ci_upper": confidence_intervals.loc["n_observables", 1],
        "n_observables_p_value": fitted_model.pvalues["n_observables"],
        "r_squared": fitted_model.rsquared,
        "status": "fitted",
    })
    return result

fraction_regression_rows = []
for template_name, template_data in fraction_analysis_data.groupby(
    "template_name", sort=True
):
    for outcome in outcome_columns:
        fraction_regression_rows.append(
            fit_fraction_regression(template_data, outcome)
        )

fraction_regression_results = pd.DataFrame(
    fraction_regression_rows,
    columns=fraction_regression_result_columns,
)



In [ ]:
millimeters_to_inches = 1 / 25.4
double_column_width = 178 * millimeters_to_inches
paper_negative_color = "#D55E00"
paper_positive_color = "#009E73"
paper_plot_style = {
    "font.size": 10,
    "text.usetex": True,
    "font.family": "serif",
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "lines.markersize": 4,
}

paper_outcome_labels = {
    "fraction_identifiable_dynamic": (
        "Identifiability (Dynamic Parameters)"
    ),
    "fraction_global_optimum": (
        "Best-Observed-Objective\nRecovery"
    ),
}

paper_figure_paths = {
    "dynamic_identifiability": Path(
        "Figures/S3A_ObservationFunctionComplexity_DynamicParameterIdentifiability.pdf"
    ),
    "best_objective_recovery": Path(
        "Figures/S3B_ObservationFunctionComplexity_BestObservedObjectiveRecovery.pdf"
    ),
}


In [ ]:
paper_fraction_predictor_labels = {
    "fraction_sums": "Ratio of summed\nobservables",
    "fraction_scales_offsets": (
        "Ratio of scale and\noffset parameters"
    ),
    "n_observables": "Number of\nobservables",
}

paper_fraction_coefficient_labels = {
    "fraction_sums": r"Coefficient $\beta_{\mathrm{sum}}$",
    "fraction_scales_offsets": (
        r"Coefficient $\beta_{\mathrm{scale/offset}}$"
    ),
    "n_observables": r"Coefficient $\beta_{\mathrm{obs}}$",
}


def plot_template_fraction_regression_coefficients(
    outcome, output_path
):
    outcome_results = (
        fraction_regression_results
        .query("outcome == @outcome and status == 'fitted'")
        .copy()
    )
    template_order = sorted(outcome_results["template_name"])

    with plt.rc_context(paper_plot_style):
        fig, axes = plt.subplots(
            1, 3,
            figsize=(
                double_column_width,
                112 * millimeters_to_inches,
            ),
            sharey=True,
        )
        for panel_index, (axis, predictor) in enumerate(zip(
            axes, fraction_predictor_columns
        )):
            coefficient_column = f"{predictor}_coefficient"
            lower_column = f"{predictor}_ci_lower"
            upper_column = f"{predictor}_ci_upper"
            plot_data = (
                outcome_results
                .set_index("template_name")
                .reindex(template_order)
                .reset_index()
            )
            y = np.arange(len(plot_data))
            coefficient = plot_data[coefficient_column]
            lower = plot_data[lower_column]
            upper = plot_data[upper_column]
            x_error = np.vstack([
                coefficient - lower,
                upper - coefficient,
            ])
            point_colors = np.where(
                coefficient < 0,
                paper_negative_color,
                paper_positive_color,
            )

            axis.errorbar(
                coefficient, y, xerr=x_error,
                fmt="none", ecolor="black",
                elinewidth=1, capsize=2.5,
            )
            axis.scatter(
                coefficient, y, s=16,
                c=point_colors, zorder=3,
            )
            axis.axvline(0, color="black", linewidth=0.8)
            axis.set_title(
                f"{chr(65 + panel_index)}) "
                f"{paper_fraction_predictor_labels[predictor]}"
            )
            axis.set_xlabel(
                paper_fraction_coefficient_labels[predictor]
            )
            axis.grid(
                axis="x", linestyle="--",
                alpha=0.5, linewidth=0.7,
            )
            axis.set_axisbelow(True)
            axis.xaxis.set_major_formatter(
                FormatStrFormatter("%g")
            )
            x_limit = 1.08 * max(
                abs(lower.min()), abs(upper.max())
            )
            axis.set_xlim(-x_limit, x_limit)

        axes[0].set_yticks(np.arange(len(template_order)))
        axes[0].set_yticklabels([
            template_name.split("_")[0]
            for template_name in template_order
        ])
        axes[0].invert_yaxis()
        fig.subplots_adjust(
            left=0.26, right=0.99, bottom=0.14,
            top=0.86, wspace=0.12,
        )
        plot_center = (
            axes[0].get_position().x0
            + axes[-1].get_position().x1
        ) / 2
        fig.suptitle(
            paper_outcome_labels[outcome].replace("\n", " "),
            x=plot_center, y=0.99,
        )
        fig.savefig(
            output_path, bbox_inches="tight",
            pad_inches=0.02,
        )
        plt.show()

    return fig


fraction_plot_outcomes = {
    "dynamic_identifiability": "fraction_identifiable_dynamic",
    "best_objective_recovery": "fraction_global_optimum",
}

fraction_template_figures = {}
for figure_name, outcome in fraction_plot_outcomes.items():
    fraction_template_figures[figure_name] = (
        plot_template_fraction_regression_coefficients(
            outcome=outcome,
            output_path=paper_figure_paths[figure_name],
        )
    )